In [18]:
fromDate = "2026-08-01"
toDate = "2026-08-31"
dataFile = r"S:\13.09.26\30909202_NEWCNEFINTRANSRPT.CSV"
taxFile = r"S:\mt- ms\Tax_Test_Data.xlsx"
output_file = 'Linking Output 8-2026.xlsx'

In [19]:
from contextlib import contextmanager
from typing import Any, Callable, Generator, Iterable, Optional
import functools
import inspect

In [20]:
import pandas as pd
pd.set_option('display.max_columns', None)

sub_types = ['beIN','beIN Quartar Installment', 'CNE Subscriber',
            'BeIN sports CC', 'beIN Bi Installment', 'Corporate Subscriber', 
            'Bein NC', 'beIN Installment Sub','beIN Dealer','Charge Back','Head End','Illegal Network','Outsiders','VIP_CNE']

invoice_types = ['Subscription Invoice']
tax_cols = ['Subscriber Number','Amount','Docinternalid','Receiptno','Created Date','Subscriber Type Name','Item Eng Name','Payment Flag Id']
tax_items = ['Sub.beiN']
exclude_jv_types = ['Debit']

payment_flags = ['Automation', 'Normal payment']


invoices_columns = ['inv_Subscriber Nr', 'inv_Doc Type', 'inv_Ftnr', 'inv_Created Date',
       'inv_Created Time', 'inv_Doc Status', 'inv_Period From',
       'inv_Period To', 'inv_Amount', 'inv_User Name',
        'inv_Default Entity Type', 'inv_Smartcard', 'inv_Bill Period',
       'inv_Bill Cycle', 'inv_Invoice Type', 'inv_Plan Name',
       'inv_Contract Number', 'inv_Subscriber Type',
       'inv_Subscriber Entity',
       'pmt_Amount', 'pmt_Ftnr', 'Docinternalid', 'flag']


In [21]:
def pair_unique_payment(df):
    unique_ftnr = df['pmt_Ftnr'].unique()
    unique_taxnr = df['Docinternalid_y'].unique()
    n = min(len(unique_ftnr), len(unique_taxnr))
    return pd.DataFrame({
        'pmt_Subscriber Nr': df.name[0],     # from groupby key
        'pmt_Doc Type': df.name[1],      # from groupby key
        'pmt_Ftnr': unique_ftnr[:n],
        'Docinternalid_y': unique_taxnr[:n]
    })


def pair_unique_invoice(df):
    unique_pmt_ftnr = df['pmt_Ftnr'].unique()
    unique_inv_ftnr = df['inv_Ftnr'].unique()
    n = min(len(unique_pmt_ftnr), len(unique_inv_ftnr))
    return pd.DataFrame({
        'inv_Subscriber Nr': df.name[0],     # from groupby key
        'inv_Doc Type': df.name[1],      # from groupby key
        'pmt_Ftnr': unique_pmt_ftnr[:n],
        'inv_Ftnr': unique_inv_ftnr[:n]
    })

In [22]:
data_src = pd.read_csv(dataFile,dtype="str", on_bad_lines='skip')
tax_source = pd.read_excel(taxFile,dtype='str')



In [23]:
tax = tax_source[tax_cols].copy()
tax['Created Date'] = pd.to_datetime(tax['Created Date'], format='%Y-%m-%d %H:%M:%S')
# tax = tax.loc[tax['Subscriber Type Name'].isin(sub_types)]
tax = tax.loc[tax['Item Eng Name']=='Sub.beiN']
tax['Amount'] = pd.to_numeric(tax['Amount'])

tax.loc[tax['Payment Flag Id'].isin(['Normal payment','Automation']),'Payment Flag Id'] ='Payment' 




print(f'tax size: {tax.shape[0]}')

tax size: 42155


In [24]:
data = data_src.copy()
data = data.loc[data['Subscriber Type'].isin(sub_types)]
data = data.loc[data['Doc Status']=='Posted']
#data = data.loc[(data['Payment Flag'].isin(payment_flags)) | data['Payment Flag'].isna()]
data = data.loc[(~data['Subscriber Nr'].str.lower().str.contains('edd')) & (~data['Subscriber Nr'].str.lower().str.contains('hu')) & (~data['Subscriber Nr'].str.lower().str.contains('be')) ]
data['Amount'] = pd.to_numeric(data['Amount'])


data['Created Date'] = pd.to_datetime(data['Created Date'], dayfirst=True)
data['Period To'] = pd.to_datetime(data['Period To'], dayfirst=True)
data['Period From'] = pd.to_datetime(data['Period From'], dayfirst=True)


## select only JULY data
data = data.loc[(data['Created Date']>= fromDate) & (data['Created Date']<= toDate)]


print(f'FT size after filters: {data.shape[0]}')

invoices = data.loc[(data["Doc Type"]=='Invoice')].copy()
invoices = invoices.loc[invoices['Invoice Type'].isin(invoice_types)]
# negative invoices
negative_invoices = invoices.loc[invoices['Amount']<0]

invoices = invoices.loc[invoices['Amount']>0]

invoices = invoices.add_prefix("inv_")
# invoices = invoices.drop_duplicates(subset='inv_Ftnr',keep='first')
print(f'invoices: {invoices.shape[0]}')



payments = data.loc[(data["Doc Type"].isin(['JV','Payment']))].copy()
payments = payments.loc[~payments['Jv Type'].isin(exclude_jv_types)]
payments = payments.add_prefix("pmt_")
print(f'payments: {payments.shape[0]}')



FT size after filters: 85072
invoices: 41687
payments: 42454


In [25]:
data

,Subscriber Nr,Doc Type,Ftnr,Created Date,Created Time,Doc Status,Period From,Period To,Bank Date,Amount,User Name,User Fullname,Payment Ref No,Event Description,Payment Batch No,Batch Approval No,Default Entity Type,Collecting Entity,Pay Mode,Jv Type,Book Number,Smartcard,Bill Period,Bill Cycle,Invoice Type,Plan Name,Contract Number,Channel Provider,Subscriber Type,Subscriber Entity,Last Four Digits Of Card,PAYMENT_FLAG,DS_PAYMENT_FLAG
86079,14779181,Debit Note,DR_749508,2026-08-01,01:15:37 PM,Posted,NaT,NaT,NaN,100.0,TFAHMY,TAMER FAHMY,NaN,PRG,NaN,NaN,CNE Head Office,NaN,NaN,NaN,NaN,10137020508,NaN,NaN,NaN,NaN,NaN,beIN,beIN Bi Installment,beIN Alex,NaN,NaN,NaN
86080,19220173,Debit Note,DR_749509,2026-08-01,05:06:34 PM,Posted,NaT,NaT,NaN,150.0,MSABRY,MAHMOUD SABRY,NaN,RM 4K,NaN,NaN,CNE Head Office,NaN,NaN,NaN,NaN,10738765105,NaN,NaN,NaN,NaN,NaN,beIN,CNE Subscriber,beIN 6 October,NaN,NaN,NaN
86081,19041304,Invoice,INV_9130055,2026-08-01,12:05:23 AM,Posted,2026-08-01,2026-10-31,NaN,1522.0,SYSADMIN,Super User,NaN,NaN,NaN,NaN,CNE Head Office,NaN,NaN,NaN,NaN,10709151228,01/08/2026 - 31/10/2026,3M,Subscription Invoice,PREMIUM 09.24,3390427,beIN,beIN Quartar Installment,CNE Head office,NaN,NaN,NaN
86082,19205993,Invoice,INV_9130096,2026-08-01,12:05:28 AM,Posted,2026-08-01,2026-10-31,NaN,1522.0,SYSADMIN,Super User,NaN,NaN,NaN,NaN,CNE Head Office,NaN,NaN,NaN,NaN,10732948889,01/08/2026 - 31/10/2026,3M,Subscription Invoice,PREMIUM 09.24,3869006,beIN,beIN Quartar Installment,EDD El Sayda,NaN,NaN,NaN
86083,19054879,Invoice,INV_9130103,2026-08-01,12:05:28 AM,Posted,2026-08-01,2026-10-31,NaN,1522.0,SYSADMIN,Super User,NaN,NaN,NaN,NaN,CNE Head Office,NaN,NaN,NaN,NaN,10709063332,01/08/2026 - 31/10/2026,3M,Subscription Invoice,PREMIUM 09.24,3867580,beIN,beIN Quartar Installment,beIN Alex,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
811270,19134308,Payment,CNEPMT_4105575,2026-08-31,11:26:56 PM,Posted,NaT,NaT,NaN,1522.0,Mobile_App,Mobile_App,1906821,525980097,NaN,NaN,CNE Head Office,CNE Head office,Cash,NaN,NaN,10719515370,NaN,NaN,NaN,NaN,NaN,beIN,beIN Quartar Installment,CNE Head office,NaN,Automation,No Flag
811271,18915901,Payment,CNEPMT_4105593,2026-08-31,11:50:32 PM,Posted,NaT,NaT,NaN,3045.0,Mobile_App,Mobile_App,1906839,525996321,NaN,NaN,CNE Head Office,CNE Head office,Cash,NaN,NaN,10655789294,NaN,NaN,NaN,NaN,NaN,beIN,CNE Subscriber,CNE Head office,NaN,Automation,No Flag
811272,17687119,Payment,CNEPMT_4105594,2026-08-31,11:51:52 PM,Posted,NaT,NaT,NaN,3045.0,Mobile_App,Mobile_App,1906840,525996754,NaN,NaN,CNE Head Office,CNE Head office,Cash,NaN,NaN,10555524791,NaN,NaN,NaN,NaN,NaN,beIN,CNE Subscriber,CNE Head office,NaN,Automation,No Flag
811273,19150279,Payment,CNEPMT_4105595,2026-08-31,11:53:28 PM,Posted,NaT,NaT,NaN,1522.0,Mobile_App,Mobile_App,1906841,525997899,NaN,NaN,CNE Head Office,CNE Head office,Cash,NaN,NaN,10730939708,NaN,NaN,NaN,NaN,NaN,beIN,beIN Quartar Installment,CNE Head office,NaN,Automation,No Flag


In [26]:
# drop duplicated invoices

# duplicated_invoices = invoices.groupby("inv_Ftnr").aggregate(count=("inv_Ftnr","count")).reset_index()
# duplicated_invoices = duplicated_invoices.loc[duplicated_invoices['count']>1]
# invoices.loc[invoices['inv_Ftnr'].isin(duplicated_invoices['inv_Ftnr'])].sort_values('inv_Ftnr').to_csv(f'duplicated_invoices.csv', index=False)

# invoices = invoices.loc[~invoices['inv_Ftnr'].isin(duplicated_invoices['inv_Ftnr'])]

invoices = invoices.drop_duplicates(subset='inv_Ftnr',keep='first')


################################################

PAYMENT - TAX linking

################################################

In [27]:
## link payments with tax
payments_merge_with_tax = pd.merge( left=payments, right=tax[['Receiptno','Docinternalid']], left_on="pmt_Ftnr", right_on="Receiptno" , how='left')

payments_not_in_tax = payments_merge_with_tax.loc[payments_merge_with_tax['Receiptno'].isna()]
payments_in_tax = payments_merge_with_tax.loc[~payments_merge_with_tax['Receiptno'].isna()]
print(f"not in tax: {payments_not_in_tax.shape[0]}")
print(f"in tax: {payments_in_tax.shape[0]}")
payments_linked = payments_merge_with_tax.loc[~payments_merge_with_tax['Receiptno'].isna()].copy()
not_linked_payments = payments_merge_with_tax.loc[payments_merge_with_tax['Receiptno'].isna()].copy()
tax_not_linked = tax.loc[~tax['Receiptno'].isin(payments['pmt_Ftnr'])]

print(f"payment no in tax: {not_linked_payments.shape[0]}")
print(f"tax no in payments: {tax_not_linked.shape[0]}")
match_with_other_fields = pd.merge(left=not_linked_payments, left_on=["pmt_Subscriber Nr", "pmt_Created Date", "pmt_Amount", "pmt_Doc Type"], 
                                   right=tax_not_linked, right_on=["Subscriber Number", "Created Date", "Amount", "Payment Flag Id"],
                                   how='inner'
                                   )
print(f'match other fields size: {match_with_other_fields.shape[0]}')
match_with_other_fields_clean = match_with_other_fields[['pmt_Subscriber Nr','pmt_Doc Type','pmt_Ftnr','Docinternalid_y']]
result = (
    match_with_other_fields_clean.groupby(['pmt_Subscriber Nr', 'pmt_Doc Type'], group_keys=False, observed=True)
      .apply(pair_unique_payment, include_groups=False)
      .reset_index(drop=True)
)
match_with_other_fields_clean = match_with_other_fields_clean.merge(right=result, how='inner')
match_with_other_fields = match_with_other_fields.merge(right=match_with_other_fields_clean,how='inner')
payments_linked_all = pd.concat([payments_linked,match_with_other_fields])
payments_linked_all['Docinternalid'] = payments_linked_all['Docinternalid'].fillna(
    payments_linked_all['Docinternalid_y']
)
payments_linked_all = payments_linked_all[['pmt_Subscriber Nr', 'pmt_Doc Type', 'pmt_Ftnr', 'pmt_Created Date',
       'pmt_Created Time', 'pmt_Doc Status', 'pmt_Period From',
       'pmt_Period To', 'pmt_Bank Date', 'pmt_Amount', 'pmt_User Name',
       'pmt_User Fullname', 'pmt_Payment Ref No', 'pmt_Event Description',
       'pmt_Payment Batch No', 'pmt_Batch Approval No',
       'pmt_Default Entity Type', 'pmt_Collecting Entity', 'pmt_Pay Mode',
       'pmt_Jv Type', 'pmt_Book Number', 'pmt_Smartcard', 'pmt_Bill Period',
       'pmt_Bill Cycle', 'pmt_Invoice Type', 'pmt_Plan Name',
       'pmt_Contract Number', 'pmt_Channel Provider', 'pmt_Subscriber Type',
       'pmt_Subscriber Entity', 'pmt_Last Four Digits Of Card',
       'pmt_PAYMENT_FLAG','Docinternalid']]

# payments_linked_all.columns = payments_linked_all.columns.str.replace('^pmt_', '', regex=True)
print(f'All linked Payments size: {payments_linked_all.shape[0]}')

not in tax: 4405
in tax: 38049
payment no in tax: 4405
tax no in payments: 4106
match other fields size: 3831
All linked Payments size: 41845


In [28]:
not_linked_payments_tax = payments.loc[~payments['pmt_Ftnr'].isin(payments_linked_all['pmt_Ftnr'])]
not_linked_payments_tax.sort_values('pmt_Subscriber Nr')

percentage = (payments_linked_all.shape[0]/payments.shape[0]) *100
print(f'Linked Payments: {payments_linked_all.shape[0]} out of {payments.shape[0]} -  {percentage}%' )

Linked Payments: 41845 out of 42454 -  98.56550619494041%


################################################

Stage 1:

DIRECT LINK INVOICES AND PAYMENTS (SAME DATE)

################################################


In [29]:
def DirectLinkSameDate(invoices_input,payments_input):
    print(f'############ Total Invoices: {invoices.shape[0]}')

    linked_invoices = invoices_input.merge(right=payments_input[['pmt_Subscriber Nr','pmt_Created Date','pmt_Amount','pmt_Ftnr']],left_on=['inv_Subscriber Nr','inv_Created Date','inv_Amount'], 
                                    right_on=['pmt_Subscriber Nr','pmt_Created Date','pmt_Amount'], how='inner')

    linked_invoices_subset = linked_invoices[['inv_Subscriber Nr','inv_Ftnr','inv_Doc Type','pmt_Ftnr']]

    result =  linked_invoices_subset.groupby(['inv_Subscriber Nr', 'inv_Doc Type'], group_keys=False, observed=True).apply(pair_unique_invoice, include_groups=False).reset_index(drop=True)

    print(f'linked invoices before clean cross duplicates: {linked_invoices.shape[0]}')

    linked_invoices = linked_invoices.merge(right=result)
    linked_invoices = linked_invoices.merge(right=payments_input[['pmt_Ftnr','Docinternalid']])

    print(f'linked invoices after clean cross duplicates: {linked_invoices.shape[0]}')


    linked_invoices.groupby("inv_Ftnr").aggregate(count=('inv_Ftnr','count')).reset_index().sort_values("count",ascending=False)
    linked_invoices['flag'] = 'One to One (Same Date)'
    linked_invoices=linked_invoices[invoices_columns]

    not_linked_invoices = invoices_input.loc[~invoices_input['inv_Ftnr'].isin(linked_invoices['inv_Ftnr'])].copy()
    not_linked_payments = payments_input.loc[~payments_input['pmt_Ftnr'].isin(linked_invoices['pmt_Ftnr'])].copy()
    print(f'############ STAGE 1')
    print(f'Not linked invoices: {not_linked_invoices.shape[0]}')
    print(f'Not linked paymnets: {not_linked_payments.shape[0]}')

    return not_linked_invoices,not_linked_payments,linked_invoices

###########################################

Stage 2:

LINK USING SUBNR, AMOUNT (NOT SAME DATE)

###########################################

In [30]:

# merge on sub,amount
def DirectLinkNotSameDate(invoices_input,payments_input,linked_invoices):
  for i in range(5):

    not_linked_payments_no_duplicates = payments_input.drop_duplicates(subset=['pmt_Subscriber Nr','pmt_Amount','pmt_Created Date'],keep='first')

    linked_invoices_tmp =  invoices_input.merge(right=not_linked_payments_no_duplicates[['pmt_Subscriber Nr','pmt_Amount','pmt_Created Date','pmt_Ftnr']], 
                                                    left_on=['inv_Subscriber Nr','inv_Amount'], right_on=['pmt_Subscriber Nr','pmt_Amount'])
    linked_invoices_tmp_subset = linked_invoices_tmp[['inv_Subscriber Nr','inv_Ftnr','inv_Doc Type','pmt_Ftnr']]

    result = (
        linked_invoices_tmp_subset.groupby(['inv_Subscriber Nr', 'inv_Doc Type'], group_keys=False, observed=True)
          .apply(pair_unique_invoice, include_groups=False)
          .reset_index(drop=True)
    )

    linked_invoices_tmp = linked_invoices_tmp.merge(result)
    linked_invoices_tmp = linked_invoices_tmp.merge(right=payments_linked_all[['pmt_Ftnr','Docinternalid']])


    linked_invoices_tmp['flag'] = 'One to One (Not Same Date)'
    linked_invoices_tmp = linked_invoices_tmp[invoices_columns]


    linked_invoices = pd.concat([linked_invoices,linked_invoices_tmp])
    not_linked_invoices = invoices.loc[~invoices['inv_Ftnr'].isin(linked_invoices['inv_Ftnr'])].copy()
    not_linked_payments = payments.loc[~payments['pmt_Ftnr'].isin(linked_invoices['pmt_Ftnr'])].copy()

  print(f'############ STAGE 2')
  print(f'Not linked invoices: {not_linked_invoices.shape[0]}')
  print(f'Not linked paymnets: {not_linked_payments.shape[0]}')
  return not_linked_invoices,not_linked_payments,linked_invoices



###########################################

Stage 3:

MANY INVOICES + ONE PAYMENT (SAME DATE)

###########################################

In [31]:

def ManyToOneSameDate(invoices_input,payments_input,linked_invoices):

    not_linked_invoices_grouped = invoices_input.groupby(['inv_Subscriber Nr','inv_Created Date']).aggregate(inv_Ftnr = ('inv_Ftnr','max'),inv_Ftnr2 = ('inv_Ftnr','min'),inv_Amount=('inv_Amount','sum'),inv_Period_From=('inv_Period From','min'),inv_Period_To=('inv_Period To','max'),inv_Bill_Cycle=('inv_Bill Cycle','sum'),inv_Ftnr_list = ('inv_Ftnr', lambda x: ','.join(x))).reset_index()

    linked_invoices_temp = not_linked_invoices_grouped.merge(right=payments_input[['pmt_Subscriber Nr','pmt_Created Date','pmt_Amount','pmt_Ftnr']],left_on=['inv_Subscriber Nr','inv_Created Date','inv_Amount'], 
                                    right_on=['pmt_Subscriber Nr','pmt_Created Date','pmt_Amount'], how='inner')


    linked_invoices_temp = linked_invoices_temp.merge(right=payments_linked_all[['pmt_Ftnr','Docinternalid']])

    inv_lst_to_pmt = linked_invoices_temp[['inv_Ftnr_list','pmt_Ftnr','Docinternalid','pmt_Amount']].copy()

    inv_lst_to_pmt['inv_Ftnr_list'] = inv_lst_to_pmt['inv_Ftnr_list'].str.split(',')

    inv_lst_to_pmt = inv_lst_to_pmt.explode('inv_Ftnr_list').reset_index(drop=True)

    linked_invoices_tmp = invoices_input.merge(right=inv_lst_to_pmt, left_on='inv_Ftnr', right_on='inv_Ftnr_list')
    linked_invoices_tmp['flag'] = 'Many to One (Same Date)'

    linked_invoices_tmp = linked_invoices_tmp[invoices_columns]
    linked_invoices = pd.concat([linked_invoices,linked_invoices_tmp])
    not_linked_invoices = invoices.loc[~invoices['inv_Ftnr'].isin(linked_invoices['inv_Ftnr'])].copy()
    not_linked_payments = payments_linked_all.loc[~payments_linked_all['pmt_Ftnr'].isin(linked_invoices['pmt_Ftnr'])].copy()

    print(f'############ STAGE 3')
    print(f'Not linked invoices: {not_linked_invoices.shape[0]}')
    print(f'Not linked paymnets: {not_linked_payments.shape[0]}')
    return not_linked_invoices,not_linked_payments,linked_invoices



###########################################

Stage 4:

MANY INVOICES + ONE PAYMENT (NOT SAME DATE)

###########################################

In [32]:
def ManyToOneNotSameDate(invoices_input,payments_input,linked_invoices):

    not_linked_invoices_grouped = invoices_input.groupby(['inv_Subscriber Nr','inv_Created Date','inv_Plan Name']).aggregate(inv_Ftnr = ('inv_Ftnr','max'),inv_Ftnr2 = ('inv_Ftnr','min'),inv_Amount=('inv_Amount','sum'),inv_Period_From=('inv_Period From','min'),inv_Period_To=('inv_Period To','max'),inv_Ftnr_list = ('inv_Ftnr', lambda x: ','.join(x))).reset_index()

    linked_invoices_temp = not_linked_invoices_grouped.merge(right=payments_input[['pmt_Subscriber Nr','pmt_Created Date','pmt_Amount','pmt_Ftnr']],left_on=['inv_Subscriber Nr','inv_Amount'], 
                                    right_on=['pmt_Subscriber Nr','pmt_Amount'], how='inner')

    linked_invoices_temp = linked_invoices_temp.merge(right=payments_input[['pmt_Ftnr','Docinternalid']])
    linked_invoices_temp

    inv_lst_to_pmt = linked_invoices_temp[['inv_Ftnr_list','pmt_Ftnr','Docinternalid','pmt_Amount']].copy()

    inv_lst_to_pmt['inv_Ftnr_list'] = inv_lst_to_pmt['inv_Ftnr_list'].str.split(',')
    inv_lst_to_pmt = inv_lst_to_pmt.explode('inv_Ftnr_list').reset_index(drop=True)

    linked_invoices_tmp = invoices_input.merge(right=inv_lst_to_pmt, left_on='inv_Ftnr', right_on='inv_Ftnr_list')
    linked_invoices_tmp['flag'] = 'Many to One (Not Same Date)'

    linked_invoices_tmp = linked_invoices_tmp[invoices_columns]

    linked_invoices = pd.concat([linked_invoices,linked_invoices_tmp])

    not_linked_invoices = invoices.loc[~invoices['inv_Ftnr'].isin(linked_invoices['inv_Ftnr'])].copy()
    not_linked_payments = payments_linked_all.loc[~payments_linked_all['pmt_Ftnr'].isin(linked_invoices['pmt_Ftnr'])].copy()

    print(f'############ STAGE 4')
    print(f'Not linked invoices: {not_linked_invoices.shape[0]}')
    print(f'Not linked paymnets: {not_linked_payments.shape[0]}')
    return not_linked_invoices,not_linked_payments,linked_invoices


####################################################

Stage 5:

Iterate through not_linked_invoices_with_payment and try to link manually

####################################################



In [33]:
def IterateOneByOne(invoices_input,payments_input,linked_invoices):

    not_linked_invoices_with_payment = invoices_input.loc[invoices_input['inv_Subscriber Nr'].isin(payments['pmt_Subscriber Nr'])].copy()
    # not_linked_invoices_with_payment.to_csv('not_linked_invoices_with_payment.csv',index=False)

    not_linked_invoices_with_payment['pmt_Ftnr'] = ""
    not_linked_invoices_with_payment['Docinternalid'] = ""
    not_linked_invoices_with_payment['pmt_Amount'] = ""



    used_payments = []

    for idx, invoice in not_linked_invoices_with_payment.iterrows():

        # Get all payments for this subscriber
        subs_payments = payments_input.loc[
            payments_input['pmt_Subscriber Nr'] == invoice['inv_Subscriber Nr']
        ].copy()



        # Exclude already used payments
        subs_payments = subs_payments.loc[
            ~subs_payments['pmt_Ftnr'].isin(used_payments)
        ]

        if subs_payments.empty:
            continue

        # Compute difference between invoice amount and payment amount
        subs_payments['diff'] = (subs_payments['pmt_Amount'] - invoice['inv_Amount']).abs()

        # Pick payment with MINIMUM difference
        best_payment = subs_payments.sort_values('diff').iloc[0]

        # Accept match only if within tolerance (±20)
        if best_payment['diff'] <= 20:
            used_payments.append(best_payment['pmt_Ftnr'])

            not_linked_invoices_with_payment.at[idx, 'pmt_Amount'] = best_payment['pmt_Amount']
            not_linked_invoices_with_payment.at[idx, 'pmt_Ftnr'] = best_payment['pmt_Ftnr']
            not_linked_invoices_with_payment.at[idx, 'Docinternalid'] = best_payment['Docinternalid']



    linked_invoices_tmp = not_linked_invoices_with_payment.loc[not_linked_invoices_with_payment['Docinternalid']!=''].copy()
    linked_invoices_tmp['flag'] = 'Iteration'

    linked_invoices = pd.concat([linked_invoices,linked_invoices_tmp])

    not_linked_invoices = invoices.loc[~invoices['inv_Ftnr'].isin(linked_invoices['inv_Ftnr'])].copy()
    not_linked_payments = payments_linked_all.loc[~payments_linked_all['pmt_Ftnr'].isin(linked_invoices['pmt_Ftnr'])].copy()

    linked_invoices = linked_invoices.drop_duplicates()

    print(f'############ STAGE 5')
    print(f'Not linked invoices: {not_linked_invoices.shape[0]}')
    print(f'Not linked paymnets: {not_linked_payments.shape[0]}')
    return not_linked_invoices,not_linked_payments,linked_invoices



In [34]:
not_linked_invoices,not_linked_payments,linked_invoices = DirectLinkSameDate(invoices_input= invoices,payments_input= payments_linked_all)
not_linked_invoices,not_linked_payments,linked_invoices = DirectLinkNotSameDate(invoices_input=not_linked_invoices, payments_input=not_linked_payments,linked_invoices=linked_invoices)
not_linked_invoices,not_linked_payments,linked_invoices = ManyToOneSameDate(invoices_input=not_linked_invoices, payments_input=not_linked_payments,linked_invoices=linked_invoices)
not_linked_invoices,not_linked_payments,linked_invoices = ManyToOneNotSameDate(invoices_input=not_linked_invoices, payments_input=not_linked_payments,linked_invoices=linked_invoices)
not_linked_invoices,not_linked_payments,linked_invoices = IterateOneByOne(invoices_input=not_linked_invoices, payments_input=not_linked_payments,linked_invoices=linked_invoices)
not_linked_invoices_with_payment = not_linked_invoices.loc[not_linked_invoices['inv_Subscriber Nr'].isin(payments['pmt_Subscriber Nr'])].copy()
not_linked_invoices_no_payment = not_linked_invoices.loc[~not_linked_invoices['inv_Subscriber Nr'].isin(payments['pmt_Subscriber Nr'])].copy()
print(f'##############################')
print(f'linked invoices: {linked_invoices.shape[0]}')
print(f'Not linked invoices with no payment: {not_linked_invoices_no_payment.shape[0]}')
print(f'Not linked invoices with payment: {not_linked_invoices_with_payment.shape[0]}')


############ Total Invoices: 41683
linked invoices before clean cross duplicates: 37745
linked invoices after clean cross duplicates: 36125
############ STAGE 1
Not linked invoices: 5558
Not linked paymnets: 5720
############ STAGE 2
Not linked invoices: 2609
Not linked paymnets: 3380
############ STAGE 3
Not linked invoices: 2136
Not linked paymnets: 2562
############ STAGE 4
Not linked invoices: 2112
Not linked paymnets: 2545
############ STAGE 5
Not linked invoices: 1494
Not linked paymnets: 1927
##############################
linked invoices: 40189
Not linked invoices with no payment: 1082
Not linked invoices with payment: 412


In [35]:
with pd.ExcelWriter(output_file) as writer:
    linked_invoices.to_excel(writer, sheet_name="Linked Invoices", index=False)
    not_linked_invoices_no_payment.to_excel(writer, sheet_name="Not linked  No Payment", index=False)
    not_linked_invoices_with_payment.to_excel(writer, sheet_name="Not Linked With Payment", index=False)
    negative_invoices.to_excel(writer, sheet_name="Negative Invoices", index=False)
    not_linked_payments.to_excel(writer, sheet_name="Not Linked Payments", index=False)


Export linked file to be splitted

In [36]:
linked_invoices.to_excel(f'to split.xlsx',index=False)

hals

In [37]:
# tax_source.loc[tax_source['Subscriber Number']=='19219638'] #19213676 # 19219638

In [38]:
# data_src.loc[data_src['Subscriber Nr']=='19219638'] 

In [39]:
# s ='19213676'
# print('linked_invoices')
# display(linked_invoices.loc[linked_invoices['inv_Subscriber Nr']==s])
# print('not_linked_invoices_with_payment')
# display(not_linked_invoices_with_payment.loc[not_linked_invoices_with_payment['inv_Subscriber Nr']==s])
# print('not_linked_invoices_no_payment')
# display(not_linked_invoices_no_payment.loc[not_linked_invoices_no_payment['inv_Subscriber Nr']==s])
# print('negative_invoices')
# display(negative_invoices.loc[negative_invoices['Subscriber Nr']==s])


In [40]:
# payments_linked.loc[payments_linked['pmt_Subscriber Nr']==s]

In [41]:
# sub = '19073383'
# print(linked_invoices.loc[linked_invoices['inv_Subscriber Nr']==sub])
# print(not_linked_invoices.loc[not_linked_invoices['inv_Subscriber Nr']==sub])
# print(not_linked_invoices_with_payment.loc[not_linked_invoices_with_payment['inv_Subscriber Nr']==sub])

In [42]:
# not_linked_invoices_with_payment.to_csv('not_linked_invoices_with_payment.csv',index = False)
# linked_invoices.to_csv('linked_invoices.csv',index=False)
# not_linked_invoices.to_csv('not_linked_invoices.csv',index=False)

In [43]:
# not_linked_invoices.loc[not_linked_invoices['inv_Subscriber Nr'].isin(payments_linked_all['pmt_Subscriber Nr'])]